# 12 · PSI / data drift

Compare a reference window (v1 train) with the final holdout on the plan's drift features. Volume is traffic, not model drift by itself.

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

from cross_model_drift.compare import DRIFT_COLUMNS
from cross_model_drift.data import load_split
from cross_model_drift.metrics import feature_psi_table
from cross_model_drift.notebook import setup_model_session

nb = setup_model_session()
reference = load_split("v1", "train", nb.config, engine=nb.engine)
current = load_split("v1", "holdout", nb.config, engine=nb.engine)
psi = feature_psi_table(reference, current, DRIFT_COLUMNS)
psi

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=psi, y="feature", x="psi", hue="kind", ax=ax)
ax.axvline(0.1, color="#F58518", linestyle="--", label="watch")
ax.axvline(0.25, color="#E45756", linestyle="--", label="shift")
ax.set_title("PSI reference (v1 train) vs holdout")
ax.legend()
nb.show(fig)
psi.to_csv(nb.artifacts / "reports" / "psi_holdout.csv", index=False)

In [ ]:
volume = nb.read_sql(
    f"""
    SELECT
        SUM(created >= '2026-05-21' AND created < '2026-07-22') AS v1_train_n,
        SUM(created >= '2026-08-05' AND created < '2026-08-22') AS holdout_n
    FROM `{nb.table}`
    """
)
volume